Assignment 3.1 – Feature Store Exercise  
AAI‑540: Machine Learning Engineering  
Author: Gregory Bauer
Date: May 2026

1. Notebook Assignment Details
---------------------------------------------------------
This notebook implements the full workflow required for Assignment 3.1.
The objective is to engineer neighborhood‑level features from the California housing dataset, enrich them with Google Maps metadata, and ingest the resulting features into Amazon SageMaker Feature Store.
The notebook concludes with Feature Store queries for three neighborhoods: Brooktree, Fisherman’s Wharf, and Los Osos.

2. Environment Setup 
---------------------------------------------------------
This section installs required dependencies and imports all libraries used throughout the notebook.
The SageMaker Feature Store SDK requires specific versions of sagemaker, boto3, and the multiprocessing libraries ppft and pathos.

In [1]:
# Install required packages for SageMaker Feature Store functionality.
# These versions are known to be compatible with the Feature Store API.
%pip install --force-reinstall --no-deps sagemaker==2.218.0
%pip install --no-deps ppft
%pip install --no-deps pathos

# Core imports for AWS, data processing, and Feature Store operations.
import boto3
import sagemaker
import pandas as pd
import numpy as np
import time

# SageMaker session and FeatureGroup class for Feature Store operations.
from sagemaker.session import Session
from sagemaker.feature_store.feature_group import FeatureGroup


  Using cached sagemaker-2.218.0-py3-none-any.whl.metadata (14 kB)
Using cached sagemaker-2.218.0-py3-none-any.whl (1.5 MB)
  Attempting uninstall: sagemaker
    Found existing installation: sagemaker 2.218.0
    Uninstalling sagemaker-2.218.0:
      Successfully uninstalled sagemaker-2.218.0
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


3. Initialize SageMaker and Feature Store Session
---------------------------------------------------------
This section initializes the SageMaker session, Feature Store runtime client, and execution role.
These objects are required to create and manage Feature Groups.

In [2]:
# Establish the active AWS region by querying the default boto3 session.
# This ensures that all subsequent SageMaker and Feature Store operations
# execute within the same regional context, avoiding cross‑region resource
# mismatches.
region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)

# Instantiate low‑level service clients for SageMaker and the Feature Store
# runtime. These clients provide direct access to the underlying service APIs
# used for feature ingestion, retrieval, and metadata management.
sagemaker_client = boto_session.client("sagemaker", region_name=region)
featurestore_runtime = boto_session.client(
    "sagemaker-featurestore-runtime",
    region_name=region
)

# Construct a Feature Store–aware SageMaker session. This wrapper integrates
# the boto3 session and service clients into a unified interface that manages
# Feature Group creation, offline/online store interactions, and S3 I/O.
feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

# Retrieve the execution role associated with the current SageMaker environment.
# This IAM role governs permissions for interacting with S3, Feature Store,
# and other AWS-managed resources used throughout the workflow.
role = sagemaker.get_execution_role()

# Identify the default S3 bucket provisioned for this SageMaker session.
# This bucket serves as the storage location for offline feature data,
# intermediate artifacts, and any Feature Store–generated outputs.
default_bucket = feature_store_session.default_bucket()

# Define a consistent S3 prefix under which all Feature Store artifacts for
# this assignment will be organized. Using a dedicated prefix improves
# traceability and prevents naming collisions across experiments.
prefix = "assignment-3-1-featurestore"


4. Load Housing and Google Maps Data 
---------------------------------------------------------
The assignment provides two datasets:

housing.csv – California housing data

google_maps_raw.csv – Google Maps metadata

These datasets are merged using latitude and longitude coordinates.

In [3]:
# Load the primary California housing dataset. This file provides the core
# structural and socioeconomic attributes (e.g., median income, room counts,
# population metrics) that form the foundation for all subsequent feature
# engineering steps in the workflow.
housing_df = pd.read_csv("housing.csv")

# Load the auxiliary dataset containing Google Maps–derived neighborhood
# metadata. This dataset enriches the housing records with spatial context,
# specifically neighborhood identifiers extracted from reverse‑geocoded
# coordinates. The filename is fixed by the assignment specification and
# must not be altered.
maps_df = pd.read_csv("housing_gmaps_data_raw.csv")

# Perform an initial structural validation by previewing the first few rows
# of each dataset. This step confirms that the files loaded correctly,
# verifies column names and data types, and ensures that both DataFrames
# are aligned with the expected schema prior to merging.
housing_df.head(), maps_df.head()


(   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
 0    -122.23     37.88                41.0        880.0           129.0   
 1    -122.22     37.86                21.0       7099.0          1106.0   
 2    -122.24     37.85                52.0       1467.0           190.0   
 3    -122.25     37.85                52.0       1274.0           235.0   
 4    -122.25     37.85                52.0       1627.0           280.0   
 
    population  households  median_income  median_house_value ocean_proximity  
 0       322.0       126.0         8.3252            452600.0        NEAR BAY  
 1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
 2       496.0       177.0         7.2574            352100.0        NEAR BAY  
 3       558.0       219.0         5.6431            341300.0        NEAR BAY  
 4       565.0       259.0         3.8462            342200.0        NEAR BAY  ,
   street_number                   route locality-political  

In [4]:
# Inspect the schema of the primary housing dataset. Printing the column names
# provides a quick structural overview and helps verify that all expected
# attributes (e.g., geographic coordinates, demographic indicators, and
# valuation metrics) are present before initiating any merge or transformation
# operations.
print(housing_df.columns)

# Examine the schema of the Google Maps–derived dataset. This check confirms
# the availability and exact spelling of neighborhood identifiers and other
# spatial metadata that will be integrated with the housing records during
# feature engineering.
print(maps_df.columns)


Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value', 'ocean_proximity'],
      dtype='object')
Index(['street_number', 'route', 'locality-political',
       'administrative_area_level_2-political',
       'administrative_area_level_1-political', 'country-political',
       'postal_code', 'address', 'longitude', 'latitude',
       'neighborhood-political', 'postal_code_suffix',
       'establishment-point_of_interest-transit_station',
       'establishment-park-point_of_interest', 'premise',
       'establishment-point_of_interest-subway_station-transit_station',
       'airport-establishment-finance-moving_company-point_of_interest-storage',
       'subpremise',
       'bus_station-establishment-point_of_interest-transit_station',
       'establishment-park-point_of_interest-tourist_attraction',
       'establishment-natural_feature',
       'airport-establishment-point_of

5. Prepare and Merge Datasets 
---------------------------------------------------------
The Google Maps dataset contains a neighborhood field named neighborhood-political.
This column is renamed to neighborhood and merged with the housing dataset using latitude and longitude.

In [5]:
# Standardize the neighborhood identifier by renaming the Google Maps field
# "neighborhood-political" to the more concise "neighborhood". This improves
# semantic clarity and ensures consistent column references throughout the
# feature‑engineering pipeline.
maps_df = maps_df.rename(columns={"neighborhood-political": "neighborhood"})

# Extract only the fields required for the spatial join. Although the Google
# Maps dataset contains additional metadata, the merge operation depends solely
# on longitude, latitude, and the derived neighborhood label. Restricting the
# DataFrame to these columns reduces memory usage and makes the merge logic
# explicit and auditable.
maps_subset = maps_df[["longitude", "latitude", "neighborhood"]]

# Integrate neighborhood metadata into the housing dataset by performing a
# left join on geographic coordinates. This operation enriches each housing
# record with its corresponding neighborhood assignment while preserving all
# original housing entries—even when a coordinate pair lacks a matching
# neighborhood in the auxiliary dataset.
merged_df = housing_df.merge(
    maps_subset,
    on=["longitude", "latitude"],
    how="left"
)

# Address missing neighborhood assignments by imputing zeros. Although
# neighborhood is conceptually categorical, the assignment specification
# requires zero‑filling to maintain numerical stability and prevent downstream
# processing errors during aggregation or feature store ingestion.
merged_df = merged_df.fillna(0)

# Conduct a brief structural validation by previewing the merged DataFrame.
# This confirms that the join executed as intended, the neighborhood field is
# present, and the resulting schema aligns with expectations for subsequent
# feature engineering steps.
merged_df.head()


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,neighborhood
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,0
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,Merriewood
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,Upper Rockridge
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,Rockridge
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,Rockridge


6. Add Primary Key and Event Time 
---------------------------------------------------------
Feature Store requires:

A unique record identifier (primary_key)

An event time column (event_time) stored as a floating‑point Unix timestamp

In [6]:
# Construct a stable primary key for Feature Store ingestion. Although the
# neighborhood label is already present, explicitly casting it to string
# guarantees a uniform dtype across all records and prevents schema‑validation
# issues when the Feature Group is created or updated.
merged_df["primary_key"] = merged_df["neighborhood"].astype(str)

# Generate a batch‑level event timestamp to satisfy Feature Store’s requirement
# for a temporal ordering field. Using time.time() yields a Unix epoch value
# with fractional‑second precision, enabling consistent ingestion semantics
# and supporting time‑aware feature retrieval in downstream ML workflows.
current_time = float(time.time())
merged_df["event_time"] = current_time


7. One‑Hot Encode Ocean Proximity 
---------------------------------------------------------
The assignment requires one‑hot encoding of the ocean_proximity column.
These engineered features will be included in the Feature Store schema.

In [7]:
# Perform a one‑hot encoding of the 'ocean_proximity' categorical attribute.
# This transformation decomposes the original nominal feature into a set of
# binary indicator variables—one per unique category—thereby enabling its use
# in numerical feature pipelines and downstream Feature Store ingestion.
# Specifying dtype=int8 minimizes memory overhead while preserving the full
# representational fidelity of the encoded categories.
ocean_dummies = pd.get_dummies(
    merged_df["ocean_proximity"],
    prefix="ocean",
    dtype="int8"
)

# Integrate the newly generated one‑hot encoded columns into the working
# DataFrame. A horizontal concatenation (axis=1) preserves the original schema
# while augmenting it with the expanded categorical representation, ensuring
# that all engineered features remain aligned on a row‑by‑row basis.
merged_df = pd.concat([merged_df, ocean_dummies], axis=1)


8. Neighborhood‑Level Feature Engineering 
---------------------------------------------------------
This section computes aggregated features at the neighborhood level:

Mean median house value (capped at 500,000)

Mean housing median age (binned in 10‑year intervals)

Mean households (rounded to nearest integer)

Bedrooms per household

In [8]:
# Aggregate housing and neighborhood‑level metrics by grouping on the
# neighborhood identifier. This operation collapses all individual housing
# records within each neighborhood and computes mean‑based summary statistics.
# These aggregated features serve as the foundational inputs for constructing
# the Feature Store’s offline and online representations.
agg_df = merged_df.groupby("neighborhood").agg({
    "median_house_value": "mean",      # Neighborhood‑level average home value
    "housing_median_age": "mean",      # Mean structural age of housing stock
    "households": "mean",              # Average household count per neighborhood
    "total_bedrooms": "mean",          # Average bedroom availability
}).reset_index()

# Apply an upper bound to the median house value to mitigate the influence of
# extreme outliers. Capping at $500,000 enforces the assignment’s constraints
# and stabilizes downstream modeling by preventing disproportionate leverage
# from high‑value neighborhoods.
agg_df["median_house_value"] = agg_df["median_house_value"].clip(upper=500000)

# Convert the continuous housing median age into interpretable 10‑year bins.
# This discretization facilitates categorical analysis and can improve model
# interpretability by grouping structurally similar neighborhoods together.
agg_df["median_house_age_bin"] = (agg_df["housing_median_age"] // 10) * 10

# Convert the mean household count into an integer representation. Although
# the aggregation produces floating‑point values, households are inherently
# discrete units; rounding restores semantic correctness for this feature.
agg_df["total_households"] = agg_df["households"].round().astype(int)

# Compute a normalized measure of bedroom availability by dividing the average
# number of bedrooms by the average number of households. This ratio provides
# a more informative indicator of housing density and living space allocation
# than raw bedroom counts alone.
agg_df["bedrooms_per_household"] = (
    agg_df["total_bedrooms"] / agg_df["households"]
)


9. Build Final Feature Store DataFrame 
---------------------------------------------------------
The final DataFrame includes all engineered features along with the required Feature Store metadata fields.

In [9]:
# Create an isolated working copy of the aggregated neighborhood‑level dataset.
# This defensive copy ensures that Feature Store–specific fields (e.g.,
# event_time, primary_key) are appended without mutating the original
# aggregation output, thereby preserving a clean baseline for validation or
# reuse in alternative workflows.
feature_df = agg_df.copy()

# Generate a batch‑level event timestamp to satisfy Feature Store’s temporal
# schema requirements. Feature Store mandates that event_time be a floating‑
# point (Fractional) Unix epoch value to support precise temporal ordering
# during ingestion and retrieval.
current_time = float(time.time())   # MUST be float for Feature Store

# Assign the event_time uniformly across all rows in this batch. Because the
# dataset represents aggregated neighborhood‑level features rather than
# time‑varying observations, a single timestamp accurately reflects the
# creation time of the entire feature batch.
feature_df["event_time"] = current_time

# Define the primary key required for Feature Store ingestion. Here, the
# neighborhood name functions as a stable, semantically meaningful unique
# identifier for each record in the Feature Group.
feature_df["primary_key"] = feature_df["neighborhood"]

# Enforce the correct dtype for event_time to ensure compatibility with the
# Feature Store schema validator. Casting to float64 guarantees consistent
# representation across all rows, even if the initial timestamp was already
# a Python float.
feature_df["event_time"] = feature_df["event_time"].astype("float64")


10. Create Feature Group (Markdown)
---------------------------------------------------------
A new Feature Group is created using a unique timestamped name.
Feature definitions are inferred directly from the DataFrame schema.

In [10]:
# Construct a globally unique Feature Group name by appending a Unix timestamp.
# Using int(time.time()) guarantees that each execution of the notebook produces
# a distinct identifier, preventing accidental collisions with previously
# registered Feature Groups that may still exist in the account.
feature_group_name = f"neighborhood-feature-group-{int(time.time())}"

# Instantiate a FeatureGroup object that will serve as the logical container
# for the engineered neighborhood‑level features. At this stage, the object is
# purely declarative—no AWS resources are created until the create() method is
# invoked later in the workflow.
neighborhood_fg = FeatureGroup(
    name=feature_group_name,
    sagemaker_session=feature_store_session
)

# Infer the Feature Store schema directly from the DataFrame. This operation
# inspects column names and dtypes in feature_df and constructs the internal
# feature definitions that will be used when registering the Feature Group.
# Importantly, this step performs only local schema preparation; it does not
# create or modify any resources in the Feature Store service.
neighborhood_fg.load_feature_definitions(data_frame=feature_df)


[FeatureDefinition(feature_name='neighborhood', feature_type=<FeatureTypeEnum.STRING: 'String'>, collection_type=None),
 FeatureDefinition(feature_name='median_house_value', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='housing_median_age', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='households', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='total_bedrooms', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='median_house_age_bin', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='total_households', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='bedrooms_per_household', feature_type=<FeatureTypeEnum.FRAC

11. Register Feature Group in SageMaker 
---------------------------------------------------------
The Feature Group is registered with both an online and offline store.
A wait loop ensures the Feature Group is fully created before ingestion begins.

In [11]:
# Define a helper routine that blocks execution until the Feature Group has
# fully transitioned out of its asynchronous creation phase. Although the
# create() call initiates provisioning, the underlying infrastructure (online
# store, offline store, and metadata resources) may take several seconds to
# materialize. Polling the FeatureGroupStatus ensures that ingestion does not
# proceed until the resource is in a stable, ready state.
def wait_for_feature_group_creation_complete(feature_group):
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for Feature Group Creation...")
        time.sleep(5)  # Introduce a delay to avoid excessive API calls
        status = feature_group.describe().get("FeatureGroupStatus")
    if status != "Created":
        # Any terminal state other than "Created" indicates that provisioning
        # failed or completed only partially, making the Feature Group unsafe
        # for ingestion. Raising an exception prevents silent data corruption.
        raise RuntimeError(f"Failed to create feature group {feature_group.name}")
    print(f"FeatureGroup {feature_group.name} successfully created.")

# Register the Feature Group with SageMaker Feature Store. This call triggers
# the creation of both the online store (DynamoDB-backed, optimized for
# low-latency lookups) and the offline store (S3-backed, optimized for
# analytical queries and historical feature retrieval). The parameters specify:
#   - s3_uri: destination for offline feature data
#   - record_identifier_name: the primary key for each record
#   - event_time_feature_name: the timestamp field used for temporal ordering
#   - role_arn: IAM role granting permissions to write to S3 and Feature Store
#   - enable_online_store: whether to provision the online feature store
neighborhood_fg.create(
    s3_uri=f"s3://{default_bucket}/{prefix}",
    record_identifier_name="primary_key",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=True,
)

# Block further execution until the Feature Group is fully provisioned. This
# synchronization step prevents ingestion attempts from failing due to the
# Feature Group still being in a transient "Creating" state.
wait_for_feature_group_creation_complete(neighborhood_fg)


Waiting for Feature Group Creation...
Waiting for Feature Group Creation...
Waiting for Feature Group Creation...
Waiting for Feature Group Creation...
FeatureGroup neighborhood-feature-group-1779459788 successfully created.


12. Ingest Features into Feature Store (Markdown)
---------------------------------------------------------
The engineered features are ingested into the Feature Store using parallel workers.

In [12]:
# Ingest the fully engineered feature DataFrame into the registered Feature Group.
#
# The ingest() operation performs a coordinated write to both the online and
# offline stores:
#   • Online store (DynamoDB): supports low‑latency, real‑time feature retrieval.
#   • Offline store (S3): maintains an immutable historical record for training,
#     auditing, and time‑travel queries.
#
# Parameters:
#   - data_frame: the DataFrame containing all engineered features, including
#     the required primary_key and event_time fields.
#   - max_workers: controls the degree of parallelism during ingestion; using
#     multiple workers accelerates throughput for larger datasets.
#   - wait=True: forces synchronous execution by blocking until all records
#     have been successfully written, ensuring downstream operations do not
#     encounter partially ingested data.
neighborhood_fg.ingest(
    data_frame=feature_df,
    max_workers=4,
    wait=True
)


IngestionManagerPandas(feature_group_name='neighborhood-feature-group-1779459788', feature_definitions={'neighborhood': {'FeatureName': 'neighborhood', 'FeatureType': 'String'}, 'median_house_value': {'FeatureName': 'median_house_value', 'FeatureType': 'Fractional'}, 'housing_median_age': {'FeatureName': 'housing_median_age', 'FeatureType': 'Fractional'}, 'households': {'FeatureName': 'households', 'FeatureType': 'Fractional'}, 'total_bedrooms': {'FeatureName': 'total_bedrooms', 'FeatureType': 'Fractional'}, 'median_house_age_bin': {'FeatureName': 'median_house_age_bin', 'FeatureType': 'Fractional'}, 'total_households': {'FeatureName': 'total_households', 'FeatureType': 'Integral'}, 'bedrooms_per_household': {'FeatureName': 'bedrooms_per_household', 'FeatureType': 'Fractional'}, 'event_time': {'FeatureName': 'event_time', 'FeatureType': 'Fractional'}, 'primary_key': {'FeatureName': 'primary_key', 'FeatureType': 'String'}}, sagemaker_fs_runtime_client_config=<botocore.config.Config obje

13. Query Feature Store 
---------------------------------------------------------
The assignment requires retrieving records for three neighborhoods:

Brooktree

Fisherman’s Wharf

Los Osos

Each query retrieves the latest feature values from the online store.

In [13]:
# Define a helper function for retrieving a single neighborhood record from the
# Feature Store’s online store. The query uses the neighborhood name as the
# primary key and returns the most recent ingested feature vector associated
# with that identifier. This enables low‑latency inspection of individual
# neighborhood‑level features directly from the online store.
def query_neighborhood(name):
    return featurestore_runtime.get_record(
        FeatureGroupName=feature_group_name,
        RecordIdentifierValueAsString=name
    )

# Retrieve the feature record for the neighborhood "Brooktree". This call
# exercises the online store lookup path and verifies that ingestion completed
# successfully for this specific primary key. The returned structure contains
# all engineered features associated with the neighborhood.
query_neighborhood("Brooktree")


{'ResponseMetadata': {'RequestId': '2fdbf3c7-8a33-429c-8191-e595906f4f59',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '2fdbf3c7-8a33-429c-8191-e595906f4f59',
   'content-type': 'application/json',
   'content-length': '876',
   'date': 'Fri, 22 May 2026 14:23:35 GMT'},
  'RetryAttempts': 0},
 'Record': [{'FeatureName': 'neighborhood', 'ValueAsString': 'Brooktree'},
  {'FeatureName': 'median_house_value', 'ValueAsString': '257400.0'},
  {'FeatureName': 'housing_median_age', 'ValueAsString': '9.0'},
  {'FeatureName': 'households', 'ValueAsString': '1438.0'},
  {'FeatureName': 'total_bedrooms', 'ValueAsString': '0.0'},
  {'FeatureName': 'median_house_age_bin', 'ValueAsString': '0.0'},
  {'FeatureName': 'total_households', 'ValueAsString': '1438'},
  {'FeatureName': 'bedrooms_per_household', 'ValueAsString': '0.0'},
  {'FeatureName': 'event_time', 'ValueAsString': '1779459788.7284942'},
  {'FeatureName': 'primary_key', 'ValueAsString': 'Brooktree'}]}

In [14]:
# Query the Feature Store for the neighborhood "Fisherman's Wharf".
# This call reuses the previously defined helper function, which performs a
# low‑latency lookup against the online store using the neighborhood name as
# the primary key. The returned structure contains the most recently ingested
# feature vector associated with this neighborhood, enabling direct inspection
# of the engineered attributes stored in the Feature Group.
query_neighborhood("Fisherman's Wharf")


{'ResponseMetadata': {'RequestId': '05a02d73-be89-46f3-92b3-32919d6a239e',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '05a02d73-be89-46f3-92b3-32919d6a239e',
   'content-type': 'application/json',
   'content-length': '896',
   'date': 'Fri, 22 May 2026 14:23:35 GMT'},
  'RetryAttempts': 0},
 'Record': [{'FeatureName': 'neighborhood',
   'ValueAsString': "Fisherman's Wharf"},
  {'FeatureName': 'median_house_value', 'ValueAsString': '500000.0'},
  {'FeatureName': 'housing_median_age', 'ValueAsString': '52.0'},
  {'FeatureName': 'households', 'ValueAsString': '250.0'},
  {'FeatureName': 'total_bedrooms', 'ValueAsString': '317.0'},
  {'FeatureName': 'median_house_age_bin', 'ValueAsString': '50.0'},
  {'FeatureName': 'total_households', 'ValueAsString': '250'},
  {'FeatureName': 'bedrooms_per_household', 'ValueAsString': '1.268'},
  {'FeatureName': 'event_time', 'ValueAsString': '1779459788.7284942'},
  {'FeatureName': 'primary_key', 'ValueAsString': "Fisherman's Wharf"

In [15]:
# Query the Feature Store for the neighborhood "Los Osos". This call leverages
# the previously defined helper function, which performs an online store lookup
# using the neighborhood name as the primary key. The returned record reflects
# the most recently ingested feature vector for this neighborhood, enabling a
# direct inspection of the engineered attributes stored in the Feature Group.
query_neighborhood("Los Osos")


{'ResponseMetadata': {'RequestId': 'e7a995ee-d410-4a9c-8279-94aa9d48e1cb',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'e7a995ee-d410-4a9c-8279-94aa9d48e1cb',
   'content-type': 'application/json',
   'content-length': '894',
   'date': 'Fri, 22 May 2026 14:23:35 GMT'},
  'RetryAttempts': 0},
 'Record': [{'FeatureName': 'neighborhood', 'ValueAsString': 'Los Osos'},
  {'FeatureName': 'median_house_value', 'ValueAsString': '221612.5'},
  {'FeatureName': 'housing_median_age', 'ValueAsString': '15.375'},
  {'FeatureName': 'households', 'ValueAsString': '611.75'},
  {'FeatureName': 'total_bedrooms', 'ValueAsString': '642.5'},
  {'FeatureName': 'median_house_age_bin', 'ValueAsString': '10.0'},
  {'FeatureName': 'total_households', 'ValueAsString': '612'},
  {'FeatureName': 'bedrooms_per_household',
   'ValueAsString': '1.0502656313853698'},
  {'FeatureName': 'event_time', 'ValueAsString': '1779459788.7284942'},
  {'FeatureName': 'primary_key', 'ValueAsString': 'Los Osos'}]

In [16]:
'''
# ============================================================
# Cleanup Cell: Remove All Assets Created During This Session
# ============================================================

import boto3
import time

# Initialize clients
sagemaker_client = boto3.client("sagemaker", region_name=region)
s3_client = boto3.client("s3", region_name=region)

# ------------------------------------------------------------
# 1. Delete the Feature Group
# ------------------------------------------------------------

print(f"Attempting to delete FeatureGroup: {feature_group_name}")

try:
    sagemaker_client.delete_feature_group(FeatureGroupName=feature_group_name)
    print("Delete request submitted. Waiting for deletion to complete...")
except Exception as e:
    print(f"FeatureGroup deletion request failed or FG already deleted: {e}")

# ------------------------------------------------------------
# 2. Wait for FeatureGroup to be fully deleted
# ------------------------------------------------------------

while True:
    try:
        status = sagemaker_client.describe_feature_group(
            FeatureGroupName=feature_group_name
        )["FeatureGroupStatus"]
        print(f"Current status: {status}. Waiting...")
        time.sleep(5)
    except sagemaker_client.exceptions.ResourceNotFound:
        print("FeatureGroup successfully deleted.")
        break

# ------------------------------------------------------------
# 3. Delete S3 offline store contents
# ------------------------------------------------------------

offline_prefix = f"{prefix}/{feature_group_name}"

print(f"Deleting S3 offline store under: s3://{default_bucket}/{offline_prefix}")

# List all objects under the prefix
objects = s3_client.list_objects_v2(
    Bucket=default_bucket,
    Prefix=offline_prefix
)

if "Contents" in objects:
    delete_keys = [{"Key": obj["Key"]} for obj in objects["Contents"]]
    s3_client.delete_objects(
        Bucket=default_bucket,
        Delete={"Objects": delete_keys}
    )
    print("S3 offline store objects deleted.")
else:
    print("No S3 objects found for offline store.")

print("Cleanup complete. Notebook can now be rerun from the top without conflicts.")


SyntaxError: incomplete input (3250611104.py, line 1)